# Example 1: Single-Gene Composition — lacZ



Compute full nucleotide composition and codon usage for the E. coli lacZ gene (V00613).



In [ ]:
# Fetch lacZ and compute its full nucleotide composition and codon usage
import os
from collections import Counter
from Bio import Entrez, SeqIO
from Bio.SeqUtils import gc_fraction

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

Entrez.email = "you@example.com"  # NCBI requires a contact email.
handle = Entrez.efetch(db="nucleotide", id="V00613", rettype="gb", retmode="text")
record = SeqIO.read(handle, "genbank")
handle.close()
SeqIO.write(record, "data/V00613.gb", "genbank")

for feature in record.features:
    if feature.type == "CDS":
        cds_seq = feature.extract(record.seq)
        break

# Full nucleotide composition
composition = Counter(str(cds_seq))
total = len(cds_seq)
lines = [f"CDS length: {total} bp"]
for base in "ACGT":
    lines.append(f"{base}: {composition[base]} ({composition[base]/total*100:.1f}%)")
lines.append(f"GC content: {gc_fraction(cds_seq)*100:.1f}%")

# Codon usage: split into non-overlapping triplets
codons = [str(cds_seq[i:i+3]) for i in range(0, len(cds_seq) - 2, 3)]
codon_counts = Counter(codons)
lines.append(f"Total codons: {len(codons)}")
lines.append("Five most common codons:")
for codon, count in codon_counts.most_common(5):
    lines.append(f"  {codon}: {count}")

print("\n".join(lines))
with open("results/example1_lacZ_composition.txt", "w") as f:
    f.write("\n".join(lines) + "\n")
print("\nSaved: data/V00613.gb, results/example1_lacZ_composition.txt")


CDS length: 3048 bp
A: 743 (24.4%)
C: 754 (24.7%)
G: 821 (26.9%)
T: 730 (24.0%)
GC content: 51.7%
Total codons: 1016
Five most common codons:
  CTG: 75
  GAA: 45
  GAT: 41
  GCC: 32
  CAG: 32

Saved: data/V00613.gb, results/example1_lacZ_composition.txt
